# Defense Demo — Bosphorus Ship Detection (Fine-tuned YOLO11s)

**What this does:** loads my fine-tuned model straight from the GitHub repository (no Drive needed), detects and **counts** ships in any image, and shows the same image **blurred / darkened / noisy / foggy** side by side.

**Defense-day usage:**
1. `Runtime > Run all` once before the defense. Takes ~3 minutes.
2. During the defense, if the jury provides an image: run the **upload cell**, then the **demo cell**. ~30 seconds.
3. The last cell saves all outputs as PNG files and offers a zip download. Keep that zip as the offline fallback.

Repository: https://github.com/Mizar-107/Uskudar-University-Thesis-Repository

In [ ]:
!pip -q install ultralytics
import ultralytics, torch
print('Ultralytics', ultralytics.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - Runtime > Change runtime type > GPU')

In [ ]:
# Download the fine-tuned weights directly from the thesis repository
import urllib.request, os
WEIGHTS_URL = 'https://github.com/Mizar-107/Uskudar-University-Thesis-Repository/raw/main/bosphorus_yolo11s_best.pt'
WEIGHTS = 'bosphorus_yolo11s_best.pt'
if not os.path.exists(WEIGHTS):
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS)
print(f'weights ready: {WEIGHTS} ({os.path.getsize(WEIGHTS)/1e6:.1f} MB)')

from ultralytics import YOLO
model = YOLO(WEIGHTS)
print('classes:', model.names)

In [ ]:
# OPTION A (live demo): upload one or more images here
from google.colab import files
up = files.upload()
DEMO_IMAGES = list(up.keys())
print('demo images:', DEMO_IMAGES)

In [ ]:
# OPTION B (pre-run): use test images from Google Drive instead of uploading
# from google.colab import drive
# import glob, os
# drive.mount('/content/drive')
# TEST_DIR = '/content/drive/MyDrive/Uskudar Uni Master/YOLO Model klasor/Test Dataset'
# candidates = sorted(glob.glob(os.path.join(TEST_DIR, '**', '*.jp*g'), recursive=True) +
#                     glob.glob(os.path.join(TEST_DIR, '**', '*.png'), recursive=True))
# print(f'{len(candidates)} test images found')
# DEMO_IMAGES = candidates[:3]

In [ ]:
# Degradation suite + detection helper
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def degrade(img, mode):
    if mode == 'original':
        return img
    if mode == 'blur':
        return cv2.GaussianBlur(img, (15, 15), 0)
    if mode == 'heavy blur':
        return cv2.GaussianBlur(img, (31, 31), 0)
    if mode == 'dark':
        return np.clip(img.astype(np.float32) * 0.35, 0, 255).astype(np.uint8)
    if mode == 'noise':
        noise = np.random.normal(0, 25, img.shape)
        return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if mode == 'fog':
        fog = np.full_like(img, 255)
        return cv2.addWeighted(img, 0.45, fog, 0.55, 0)
    raise ValueError(mode)

def detect(img_bgr, conf=0.25):
    r = model.predict(source=img_bgr, conf=conf, imgsz=1088, verbose=False)[0]
    n = len(r.boxes)
    confs = r.boxes.conf.cpu().numpy() if n else np.array([])
    return r.plot(), n, confs

CONDITIONS = ['original', 'blur', 'heavy blur', 'dark', 'noise', 'fog']
os.makedirs('demo_outputs', exist_ok=True)
print('helpers ready')

In [ ]:
# THE DEMO — detect on the original and every degraded version
import pandas as pd
rows = []
for path in DEMO_IMAGES:
    img = cv2.imread(path)
    if img is None:
        print('cannot read', path); continue
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    fig.suptitle(os.path.basename(path), fontsize=13)
    for ax, cond in zip(axes.ravel(), CONDITIONS):
        annotated, n, confs = detect(degrade(img, cond))
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        extra = f', mean conf {confs.mean():.2f}' if n else ''
        ax.set_title(f'{cond}: {n} ship(s){extra}', fontsize=11)
        ax.axis('off')
        rows.append({'image': os.path.basename(path), 'condition': cond,
                     'ships detected': n,
                     'mean confidence': round(float(confs.mean()), 3) if n else None})
    plt.tight_layout()
    out = os.path.join('demo_outputs', os.path.splitext(os.path.basename(path))[0] + '_demo.png')
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()

summary = pd.DataFrame(rows).pivot(index='image', columns='condition', values='ships detected')[CONDITIONS]
summary.to_csv('demo_outputs/demo_summary.csv')
print('\nShips detected per condition:')
summary

In [ ]:
# Fallback pack: zip all demo outputs and download
import shutil
shutil.make_archive('demo_outputs_pack', 'zip', 'demo_outputs')
from google.colab import files as gfiles
gfiles.download('demo_outputs_pack.zip')

## Speaking notes for the demo

1. Clean image first: "The fine-tuned model detects and counts every vessel, about 33 milliseconds per frame at 1088 input."
2. Degraded grid: "The same image under blur, low light, noise, and simulated fog. Training augmentation gives partial robustness to moderate degradation."
3. If counts drop under heavy degradation, say it plainly and connect it to the thesis: "A systematic degraded-input evaluation was outside the protocol; Section 6.2 lists it as future work. What you see is a qualitative demonstration."

Never claim a number you have not measured. Every claim in the thesis is backed; keep the demo in the same spirit.